# Practice 4-6. 앙상블 검색 (Ensemble Retrieval) — BM25 + pgvector 밀집 검색

앙상블 검색은 앞서 살펴본 희소 검색(BM25)과 밀집 검색을 함께 쓰는 방식입니다. 희소 검색은 키워드가 정확히 일치하는
문서를 빠르게 찾지만 표현이 다르면 놓치고, 밀집 검색은 의미가 비슷한 문서를 찾지만 정확한 숫자·고유명사 매칭에는
오히려 약할 수 있습니다. 두 방식을 함께 쓰면 서로의 약점을 보완할 수 있습니다.

책은 `04-postgresql-bm25-sparse-retrieval.ipynb`/`05-pgvector-hnsw-dense-retrieval.ipynb`와 같은 코드로 `bm25_retriever`·`faiss_retriever`를 다시 만든 뒤 `EnsembleRetriever`로
묶습니다. 이 노트북도 동일한 구조로, `04-postgresql-bm25-sparse-retrieval.ipynb`의 트리거 기반 BM25 인프라와 `05-pgvector-hnsw-dense-retrieval.ipynb`의 pgvector HNSW 밀집 인덱스를
**이 노트북 하나에서 처음부터 다시 구축**한 뒤(두 노트북을 먼저 실행해 두지 않아도 되도록) `EnsembleRetriever`로 묶습니다.

## 0. 환경 설정

In [ ]:
import os
import re
import psycopg
from urllib.parse import quote_plus

# --- LM Studio ---
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://...:.../v1")
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"   # LM Studio에 로드되어 있는 로컬 채팅 모델

# --- PostgreSQL (pgvector 이미지, Docker 컨테이너) ---
PG_HOST = os.getenv("PG_HOST", "pgvector")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB = os.getenv("PG_DB", "admin")
PG_USER = os.getenv("PG_USER", "admin")
PG_PASSWORD = os.getenv("PG_PASSWORD", "...")
PG_DSN = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
PG_ASYNC_URL = f"postgresql+asyncpg://{PG_USER}:{quote_plus(PG_PASSWORD)}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "투자설명서.pdf"


def strip_think(text: str) -> str:
    """qwen3 등 추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    # qwen3 계열 추론 모델은 답변 전에 <think> 추론 토큰을 많이 소모하므로 넉넉히 잡는다
    max_tokens=2048,
)

base_embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print("PostgreSQL:", cur.fetchone()[0].split(",")[0])

EMBED_DIM = len(base_embeddings.embed_query("연결 테스트"))
print("EMBED:", EMBED_DIM, "차원")
print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])


## 1. 문서 로드 · 분할

앞선 희소 검색·밀집 검색 노트북과 동일하게 `투자설명서.pdf`를 `chunk_size=2000, chunk_overlap=200`으로 분할합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(DATA_PATH)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = loader.load_and_split(doc_splitter)
print("분할된 문서 수:", len(docs))

QUESTION = "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"


## 2. 희소 검색(BM25) 인프라 구축

`04-postgresql-bm25-sparse-retrieval.ipynb`와 동일한 스키마·트리거·BM25 함수입니다(자세한 설명은 `04-postgresql-bm25-sparse-retrieval.ipynb` 참고). `DELETE` 후 재적재하므로
재실행해도 안전합니다.

In [ ]:
DDL_BM25_ALL = """
CREATE EXTENSION IF NOT EXISTS textsearch_ko;

CREATE TABLE IF NOT EXISTS practice4_bm25_docs (
    id       SERIAL PRIMARY KEY,
    content  TEXT NOT NULL,
    page     INTEGER,
    tsv      TSVECTOR,
    doc_len  INTEGER
);
CREATE INDEX IF NOT EXISTS idx_practice4_bm25_docs_tsv
    ON practice4_bm25_docs USING GIN (tsv);

CREATE TABLE IF NOT EXISTS practice4_bm25_term_freq (
    doc_id  INTEGER NOT NULL REFERENCES practice4_bm25_docs(id) ON DELETE CASCADE,
    token   TEXT NOT NULL,
    tf      INTEGER NOT NULL,
    PRIMARY KEY (doc_id, token)
);
CREATE INDEX IF NOT EXISTS idx_practice4_bm25_term_freq_token
    ON practice4_bm25_term_freq (token);

CREATE TABLE IF NOT EXISTS practice4_bm25_term_df (
    token  TEXT PRIMARY KEY,
    df     INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS practice4_bm25_stats (
    id         SMALLINT PRIMARY KEY DEFAULT 1,
    doc_count  INTEGER NOT NULL DEFAULT 0,
    total_len  BIGINT  NOT NULL DEFAULT 0,
    CHECK (id = 1)
);
INSERT INTO practice4_bm25_stats (id, doc_count, total_len)
VALUES (1, 0, 0)
ON CONFLICT (id) DO NOTHING;

CREATE OR REPLACE FUNCTION practice4_bm25_docs_before() RETURNS TRIGGER AS $$
BEGIN
    NEW.tsv := to_tsvector('korean', NEW.content);
    SELECT COALESCE(SUM(cardinality(positions)), 0)
      INTO NEW.doc_len
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights);
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_before
    BEFORE INSERT OR UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_before();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_insert() RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count + 1,
           total_len = total_len + NEW.doc_len
     WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_insert
    AFTER INSERT ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_insert();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_delete() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count - 1,
           total_len = total_len - OLD.doc_len
     WHERE id = 1;
    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_delete
    AFTER DELETE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_delete();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_update() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats SET doc_count = doc_count - 1, total_len = total_len - OLD.doc_len WHERE id = 1;

    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats SET doc_count = doc_count + 1, total_len = total_len + NEW.doc_len WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_update
    AFTER UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_update();

CREATE OR REPLACE FUNCTION practice4_bm25_search(
    q       text,
    k       integer DEFAULT 5,
    bm25_k1 float8  DEFAULT 1.2,
    bm25_b  float8  DEFAULT 0.75
) RETURNS TABLE(doc_id integer, content text, page integer, score float8) AS $$
    WITH stats AS (
        SELECT doc_count::float8 AS n,
               CASE WHEN doc_count > 0 THEN total_len::float8 / doc_count ELSE 0 END AS avgdl
          FROM practice4_bm25_stats WHERE id = 1
    ),
    qtokens AS (
        SELECT DISTINCT lexeme AS token
          FROM unnest(to_tsvector('korean', q)) AS u(lexeme, positions, weights)
    ),
    qidf AS (
        SELECT qt.token,
               ln( (s.n - COALESCE(d.df, 0) + 0.5) / (COALESCE(d.df, 0) + 0.5) + 1 ) AS idf
          FROM qtokens qt
          CROSS JOIN stats s
          LEFT JOIN practice4_bm25_term_df d ON d.token = qt.token
    ),
    scored AS (
        SELECT tf.doc_id,
               SUM( qidf.idf * (tf.tf * (bm25_k1 + 1))
                    / (tf.tf + bm25_k1 * (1 - bm25_b + bm25_b * doc.doc_len / NULLIF(s.avgdl, 0)))
                  ) AS score
          FROM practice4_bm25_term_freq tf
          JOIN qidf ON qidf.token = tf.token
          JOIN practice4_bm25_docs doc ON doc.id = tf.doc_id
          CROSS JOIN stats s
         GROUP BY tf.doc_id
    )
    SELECT scored.doc_id, doc.content, doc.page, scored.score
      FROM scored
      JOIN practice4_bm25_docs doc ON doc.id = scored.doc_id
     ORDER BY scored.score DESC
     LIMIT k;
$$ LANGUAGE sql STABLE;
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_BM25_ALL)
print("BM25 스키마 준비 완료")


In [ ]:
def reload_bm25_docs():
    """재실행 안전: DELETE → 트리거가 파생 테이블(순색인/역색인/통계)을 원복 → 재적재."""
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("DELETE FROM practice4_bm25_docs;")
            cur.execute("ALTER SEQUENCE practice4_bm25_docs_id_seq RESTART WITH 1;")
            for d in docs:
                cur.execute(
                    "INSERT INTO practice4_bm25_docs (content, page) VALUES (%s, %s)",
                    (d.page_content, d.metadata.get("page")),
                )
        conn.commit()

    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT doc_count, total_len FROM practice4_bm25_stats;")
            return cur.fetchone()


bm25_doc_count, bm25_total_len = reload_bm25_docs()
assert bm25_doc_count == len(docs), "BM25 적재 건수가 분할 문서 수와 다릅니다"
print("BM25 적재 완료:", bm25_doc_count, "건 (재실행 시에도 항상 이 값으로 재구축됩니다)")


## 3. 밀집 검색 인프라 구축 (pgvector HNSW)

In [ ]:
from langchain_core.embeddings import Embeddings

PROJ_DIM = 1024  # pgvector HNSW(vector) 인덱스 상한(2000차원)보다 충분히 작게


class TruncatedEmbeddings(Embeddings):
    """LM Studio 임베딩(embedding-8b:sl, MRL로 학습된 Qwen3 계열)의 앞쪽 out_dim개 차원만 사용해
    pgvector HNSW 인덱스 상한 안으로 맞추는 래퍼 (MRL truncation, 자세한 설명은 05-pgvector-hnsw-dense-retrieval.ipynb 참고)."""

    def __init__(self, base: Embeddings, out_dim: int):
        self.base = base
        self.out_dim = out_dim

    def embed_documents(self, texts: list) -> list:
        return [v[: self.out_dim] for v in self.base.embed_documents(texts)]

    def embed_query(self, text: str) -> list:
        return self.base.embed_query(text)[: self.out_dim]


embeddings = TruncatedEmbeddings(base_embeddings, out_dim=PROJ_DIM)

from langchain_postgres import PGEngine, PGVectorStore
from langchain_postgres.v2.indexes import HNSWIndex, DistanceStrategy

DENSE_TABLE = "practice4_dense_docs"

engine = PGEngine.from_connection_string(url=PG_ASYNC_URL)


def rebuild_dense_table():
    """재실행 안전: DROP TABLE IF EXISTS 후 재생성 → 재적재 → HNSW 재인덱싱."""
    engine.init_vectorstore_table(
        table_name=DENSE_TABLE,
        vector_size=PROJ_DIM,
        overwrite_existing=True,
    )
    vs = PGVectorStore.create_sync(
        engine=engine,
        embedding_service=embeddings,
        table_name=DENSE_TABLE,
        distance_strategy=DistanceStrategy.COSINE_DISTANCE,
        k=2,
    )
    vs.add_documents(docs)
    vs.apply_vector_index(
        HNSWIndex(name="practice4_dense_docs_hnsw", distance_strategy=DistanceStrategy.COSINE_DISTANCE, m=16, ef_construction=64)
    )
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT count(*) FROM {DENSE_TABLE};")
            row_count = cur.fetchone()[0]
    return vs, row_count


vectorstore, dense_doc_count = rebuild_dense_table()
assert dense_doc_count == len(docs), "밀집 검색 적재 건수가 분할 문서 수와 다릅니다"
print("밀집 검색 인덱스 준비 완료:", dense_doc_count, "건, HNSW 유효성:", vectorstore.is_valid_index("practice4_dense_docs_hnsw"))


### 3.1 재실행 안전성 직접 증명 — 두 인프라를 한 번씩 더 초기화·재구축

BM25 쪽은 `DELETE`(트리거가 순색인·역색인·통계를 원복), 밀집 검색 쪽은 `DROP TABLE IF EXISTS` 후 재생성으로 초기화합니다.
말로만 안전하다고 하지 않고, 두 인프라를 한 번씩 더 재구축해서 값이 그대로인지 확인합니다.

In [ ]:
bm25_doc_count_2nd, bm25_total_len_2nd = reload_bm25_docs()
vectorstore, dense_doc_count_2nd = rebuild_dense_table()

print("BM25      : 1차", (bm25_doc_count, bm25_total_len), " / 2차", (bm25_doc_count_2nd, bm25_total_len_2nd))
print("밀집 검색 : 1차", dense_doc_count, " / 2차", dense_doc_count_2nd)

assert (bm25_doc_count, bm25_total_len) == (bm25_doc_count_2nd, bm25_total_len_2nd)
assert dense_doc_count == dense_doc_count_2nd
print("\n두 인프라 모두 재구축 결과가 이전과 정확히 일치합니다 — 이 노트북은 몇 번을 다시 실행해도 안전합니다.")


## 4. 앙상블 리트리버로 결합

`04-postgresql-bm25-sparse-retrieval.ipynb`의 BM25 리트리버(`BaseRetriever` 상속)와 `05-pgvector-hnsw-dense-retrieval.ipynb`의 pgvector 밀집 리트리버(`as_retriever()`)를
`EnsembleRetriever`로 묶습니다. 책과 동일하게 가중치는 0.5 : 0.5로 둡니다.

In [ ]:
from typing import List
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun


class PgSqlFunctionRetriever(BaseRetriever):
    """PostgreSQL의 practice4_bm25_search() 함수를 호출해 검색하는 리트리버 (04-postgresql-bm25-sparse-retrieval.ipynb 참고)."""

    dsn: str
    sql_func: str
    k: int = 4

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        with psycopg.connect(self.dsn) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    f"SELECT doc_id, content, page, score FROM {self.sql_func}(%s, %s)",
                    (query, self.k),
                )
                rows = cur.fetchall()
        return [
            Document(
                page_content=content,
                metadata={"source": DATA_PATH, "doc_id": doc_id, "page": page, "score": float(score)},
            )
            for doc_id, content, page, score in rows
        ]


bm25_retriever = PgSqlFunctionRetriever(dsn=PG_DSN, sql_func="practice4_bm25_search", k=2)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

try:
    from langchain_classic.retrievers import EnsembleRetriever
except ImportError:
    from langchain.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)
print("ensemble_retriever 준비 완료")


## 5. 검색 확인

In [ ]:
found = ensemble_retriever.invoke(QUESTION)
print(f"앙상블 검색 결과 {len(found)}건")
for d in found:
    print(" -", d.metadata.get("page"), "|", d.page_content[:50])


## 6. 답변 생성 (RetrievalQA)

In [ ]:
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True,
)

result = qa_chain.invoke({"query": QUESTION})

print("질문:", QUESTION)
print("답변:", strip_think(result["result"]))
print("\n사용된 문서:")
for d in result["source_documents"]:
    print(" -", d.metadata)


## 7. 관찰 — 앙상블은 "둘 중 하나만 맞아도" 산다

`04-postgresql-bm25-sparse-retrieval.ipynb`의 BM25(`k=2`)는 "총 발행량"이라는 표현과 겹치는 단어가 적어 정답 문단을 놓쳤습니다. 반면 `05-pgvector-hnsw-dense-retrieval.ipynb`의
밀집 검색(`k=2`)은 1위로 정답 문단(328쪽, "Ⅳ. 발행주식의 총수 ... 13,602,977")을 정확히 찾아냈습니다. `EnsembleRetriever`는
두 리트리버의 결과를 순위 기반으로 합치므로(RRF, Reciprocal Rank Fusion), **밀집 검색 쪽에서 찾아낸 정답 문단이 앙상블
결과에도 살아남아** 위 답변에 그대로 반영됩니다.

다만 이 방식의 한계도 분명합니다 — `EnsembleRetriever`는 각 리트리버가 이미 잘라낸 상위 k건만 합칠 뿐, **두 리트리버
모두의 상위 k건에서 빠진 문서는 앙상블도 복구하지 못합니다.** 즉 앙상블의 재현율은 "개별 리트리버 중 최선의 재현율"을
넘지 못하고, 그 이하로 떨어지지도 않게 두 신호를 섞어주는 것에 가깝습니다. 실무에서는 각 리트리버의 k(또는 후보 수)를
최종 반환 개수보다 넉넉히 크게 잡아 합친 뒤 재순위화(rerank)하는 방식으로 이 한계를 완화합니다.

## 8. 정리 — 이 세 노트북이 book의 예제와 다른 점

| | 책 (FAISS/BM25Retriever) | 이 노트북들 (pgvector) |
|---|---|---|
| 스파스 검색 | 프로세스 메모리 내 BM25 (Kiwi 토큰화) | PostgreSQL 트리거로 증분 유지되는 순색인/역색인 + SQL로 계산하는 BM25 |
| 밀집 검색 | FAISS 인메모리 인덱스, 파일로 save/load | pgvector HNSW 인덱스, DB에 영구 저장 |
| 앙상블 | `EnsembleRetriever(bm25, faiss)` | `EnsembleRetriever(bm25_retriever, dense_retriever)` — 리트리버 인터페이스는 동일 |
| 확장성 | 검색기마다 코퍼스 전체를 프로세스 메모리에 적재 | 색인·데이터가 전부 PostgreSQL 서버에 있어 여러 프로세스가 공유, 문서 증가는 트리거의 증분 갱신·ANN 인덱스로 흡수 |

`BaseRetriever`/`EnsembleRetriever` 같은 LangChain의 인터페이스는 그대로 유지하면서, 실제 저장·색인·랭킹 연산만
전부 PostgreSQL(pgvector) 서버로 옮긴 것이 이 세 노트북의 핵심입니다.